# Multi-Timeframe Strategy Demonstration

This notebook recreates `qc_projects/main_multi_timeframe.py`, our Lean algorithm that:
- Filters for an uptrend when the daily `SMA(50)` exceeds `SMA(200)`
- Uses hourly `RSI(14)` for tactical entries when the market is oversold
- Exits when RSI crosses 70 or the daily trend flips negative

Below we reproduce the same data stack with Yahoo Finance so we can iterate locally before syncing changes back to QuantConnect.

## Step 1: Imports and Parameters

We mirror the Lean configuration: SPY universe, 2020-01-01 → 2022-01-01 test window, RSI thresholds 30/70, and a 210-day warmup to seed the long SMAs.

In [8]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

pd.set_option("display.max_columns", None)

symbol = "SPY"
start_date = "2020-01-01"
end_date = "2022-01-01"
sma_fast = 50
sma_slow = 200
rsi_period = 14
rsi_oversold = 30
rsi_exit = 70
warmup_days = 210
max_hourly_lookback_days = 700

today = pd.Timestamp.today().tz_localize(None)
requested_start = pd.Timestamp(start_date)
requested_end = pd.Timestamp(end_date)

hourly_start = max(requested_start, today - pd.Timedelta(days=max_hourly_lookback_days))
hourly_end = min(requested_end, today)

print(f"Symbol: {symbol} | Requested window: {start_date} → {end_date}")
print(f"Trend filter: SMA{sma_fast}/{sma_slow} | RSI: {rsi_period}-period Welles Wilder")
print(f"Hourly data window (per Yahoo limits): {hourly_start.date()} → {hourly_end.date()}")

Symbol: SPY | Requested window: 2020-01-01 → 2022-01-01
Trend filter: SMA50/200 | RSI: 14-period Welles Wilder
Hourly data window (per Yahoo limits): 2024-02-28 → 2022-01-01


## Step 2: Download Daily OHLCV for Trend Filter

Following the other notebooks, we use Yahoo Finance with `multi_level_index=False` and compute the long-term SMAs on the daily closes.

In [9]:
daily_df = yf.download(
    symbol,
    start=start_date,
    end=end_date,
    interval="1d",
    progress=False,
    multi_level_index=False,
    auto_adjust=False,
)
daily_df = daily_df.dropna().tz_localize(None)
daily_df.index.name = "date"

daily_df.head()

,Adj Close,Close,High,Low,Open,Volume
date,,,,,,
2020-01-02,297.698914,324.869995,324.890015,322.529999,323.540009,59151200
2020-01-03,295.444763,322.410004,323.640015,321.100006,321.160004,77709700
2020-01-06,296.571869,323.640015,323.730011,320.359985,320.489990,55653900
2020-01-07,295.738037,322.730011,323.540009,322.239990,323.019989,40496400
2020-01-08,297.314148,324.450012,325.779999,322.670013,322.940002,68296000


## Step 3: Build Daily Trend Filter (SMA 50/200)

Lean keeps two rolling Simple Moving Averages on the daily close. We'll compute them here and derive a boolean `uptrend` flag used later in the hourly loop.

In [10]:
daily_features = daily_df[["Close"]].copy()
daily_features["sma_fast"] = daily_features["Close"].rolling(sma_fast).mean()
daily_features["sma_slow"] = daily_features["Close"].rolling(sma_slow).mean()
daily_features["uptrend"] = daily_features["sma_fast"] > daily_features["sma_slow"]

daily_features.dropna(inplace=True)
daily_features = daily_features.loc[
    (daily_features.index >= hourly_start.normalize()) &
    (daily_features.index <= hourly_end.normalize())
]
daily_features.tail()

,Close,sma_fast,sma_slow,uptrend
date,,,,


## Step 4: Download Hourly Bars for RSI Timing

Yahoo Finance only allows ~730 days of 60-minute history per request. We therefore (1) cap the hourly window to the most recent ~700 days within the requested range, and (2) download that span in manageable chunks while keeping `multi_level_index=False` for consistency.

In [11]:
def download_hourly_in_chunks(symbol: str, start: pd.Timestamp, end: pd.Timestamp, chunk_days: int = 700) -> pd.DataFrame:
    start_ts = start
    end_ts = end
    frames = []
    cursor = start_ts
    chunk_delta = pd.Timedelta(days=chunk_days)

    while cursor < end_ts:
        chunk_end = min(cursor + chunk_delta, end_ts)
        data = yf.download(
            symbol,
            start=cursor,
            end=chunk_end,
            interval="60m",
            progress=False,
            multi_level_index=False,
            auto_adjust=False,
        )
        if data is not None and not data.empty:
            frames.append(data)
        else:
            print(f"Warning: no data for chunk {cursor.date()} → {chunk_end.date()}")
        cursor = chunk_end

    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames)
    combined = combined[~combined.index.duplicated(keep="last")]
    return combined

hourly_df = download_hourly_in_chunks(symbol, hourly_start, hourly_end)
if hourly_df.empty:
    raise RuntimeError(
        "No hourly data retrieved. Try increasing end_date towards today or using a shorter window within the last ~700 days."
    )

hourly_df = hourly_df.dropna().tz_localize(None)
hourly_df.index.name = "timestamp"

print(f"Hourly bars fetched: {len(hourly_df):,}")
hourly_df.head()

RuntimeError: No hourly data retrieved. Try increasing end_date towards today or using a shorter window within the last ~700 days.

## Step 5: Compute Hourly RSI and Merge Daily Trend State

We reproduce the Lean RSI (Wilder smoothing) and join each hourly bar with the same-day SMA signals so the entry logic can inspect both timeframes simultaneously.

In [ ]:
def wilder_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

hourly_features = hourly_df.copy()
hourly_features["rsi"] = wilder_rsi(hourly_features["Close"], period=rsi_period)
hourly_features["session_date"] = hourly_features.index.floor("D")

trend_columns = daily_features[["sma_fast", "sma_slow", "uptrend"]]
hourly_features = hourly_features.merge(
    trend_columns,
    left_on="session_date",
    right_index=True,
    how="left",
)

hourly_features.dropna(subset=["rsi", "sma_slow"], inplace=True)
hourly_features.head()

## Step 6: Replay the Hourly Entry/Exit Logic

Lean enters when the daily trend is bullish and the hourly RSI dips below 30, then exits when RSI exceeds 70 or the trend flips. We transcribe that state machine here.

In [ ]:
positions = []
trades = []
entry_markers = []
exit_markers = []
active_position = None

for ts, row in hourly_features.iterrows():
    uptrend = bool(row["uptrend"])
    rsi_value = row["rsi"]
    price = row["Close"]

    if active_position is None:
        if uptrend and rsi_value < rsi_oversold:
            active_position = {
                "entry_time": ts,
                "entry_price": price,
                "entry_rsi": rsi_value,
            }
            entry_markers.append({"timestamp": ts, "price": price})
    else:
        exit_signal = False
        reason = None
        if not uptrend:
            exit_signal = True
            reason = "Trend reversal"
        elif rsi_value > rsi_exit:
            exit_signal = True
            reason = "RSI overbought"

        if exit_signal:
            ret_pct = (price - active_position["entry_price"]) / active_position["entry_price"] * 100
            trades.append({
                "entry_time": active_position["entry_time"],
                "exit_time": ts,
                "entry_price": round(active_position["entry_price"], 2),
                "exit_price": round(price, 2),
                "entry_rsi": round(active_position["entry_rsi"], 2),
                "exit_rsi": round(rsi_value, 2),
                "return_pct": round(ret_pct, 2),
                "reason": reason,
            })
            exit_markers.append({"timestamp": ts, "price": price})
            active_position = None

# Force liquidation at the final bar if still invested
if active_position is not None:
    last_price = hourly_features.iloc[-1]["Close"]
    last_time = hourly_features.index[-1]
    ret_pct = (last_price - active_position["entry_price"]) / active_position["entry_price"] * 100
    trades.append({
        "entry_time": active_position["entry_time"],
        "exit_time": last_time,
        "entry_price": round(active_position["entry_price"], 2),
        "exit_price": round(last_price, 2),
        "entry_rsi": round(active_position["entry_rsi"], 2),
        "exit_rsi": round(hourly_features.iloc[-1]["rsi"], 2),
        "return_pct": round(ret_pct, 2),
        "reason": "End of data",
    })
    exit_markers.append({"timestamp": last_time, "price": last_price})
    active_position = None

trades_df = pd.DataFrame(trades)
print(f"Completed simulation with {len(trades_df)} trades")
trades_df.head() if not trades_df.empty else "No qualifying entries"

## Step 7: Aggregate Performance Metrics

Quick sanity checks before pushing settings back to Lean: trade count, win rate, average return, and holding duration.

In [ ]:
if trades_df.empty:
    summary = {
        "total_trades": 0,
        "win_rate_pct": 0.0,
        "avg_return_pct": 0.0,
        "median_return_pct": 0.0,
        "avg_hold_hours": 0.0,
    }
else:
    hold_hours = (pd.to_datetime(trades_df["exit_time"]) - pd.to_datetime(trades_df["entry_time"])).dt.total_seconds() / 3600
    summary = {
        "total_trades": len(trades_df),
        "win_rate_pct": round((trades_df["return_pct"] > 0).mean() * 100, 1),
        "avg_return_pct": round(trades_df["return_pct"].mean(), 2),
        "median_return_pct": round(trades_df["return_pct"].median(), 2),
        "avg_hold_hours": round(hold_hours.mean(), 1),
    }

summary

## Step 8: Visualize Price, Trend Filter, and RSI Signals

We plot daily price with SMA overlays plus entry/exit markers, alongside hourly RSI to confirm the timing. Figures are saved to `graphs/` for side-by-side comparison with other notebooks.

In [ ]:
graphs_dir = Path("graphs")
graphs_dir.mkdir(exist_ok=True)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=("Daily Trend + Signals", "Hourly RSI"),
)

fig.add_trace(
    go.Scatter(x=daily_features.index, y=daily_features["Close"], name="Daily Close", line=dict(color="#1f77b4")),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=daily_features.index, y=daily_features["sma_fast"], name=f"SMA{sma_fast}", line=dict(color="#ff7f0e")),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=daily_features.index, y=daily_features["sma_slow"], name=f"SMA{sma_slow}", line=dict(color="#2ca02c")),
    row=1,
    col=1,
)

if entry_markers:
    fig.add_trace(
        go.Scatter(
            x=[m["timestamp"] for m in entry_markers],
            y=[m["price"] for m in entry_markers],
            mode="markers",
            marker=dict(symbol="triangle-up", size=10, color="#2ca02c"),
            name="Entries",
        ),
        row=1,
        col=1,
    )
if exit_markers:
    fig.add_trace(
        go.Scatter(
            x=[m["timestamp"] for m in exit_markers],
            y=[m["price"] for m in exit_markers],
            mode="markers",
            marker=dict(symbol="triangle-down", size=10, color="#d62728"),
            name="Exits",
        ),
        row=1,
        col=1,
    )

fig.add_trace(
    go.Scatter(x=hourly_features.index, y=hourly_features["rsi"], name="RSI", line=dict(color="#9467bd")),
    row=2,
    col=1,
)
fig.add_hline(y=rsi_oversold, line=dict(color="#2ca02c", dash="dash"), row=2, col=1)
fig.add_hline(y=rsi_exit, line=dict(color="#d62728", dash="dash"), row=2, col=1)

fig.update_layout(height=700, title="Multi-Timeframe Strategy Signals")
output_path = graphs_dir / "multi_timeframe_signals.html"
fig.write_html(output_path)
fig.show()
print(f"Saved interactive visualization to {output_path}")

## Step 9: Inspect Trade Log

Lean would emit `Debug` statements; here we keep the structured dataframe for further analysis or export.

In [ ]:
if trades_df.empty:
    print("No trades were generated under the current configuration.")
else:
    display(trades_df.tail(10))

## Recap

- Daily SMAs and hourly RSI follow the exact structure from `qc_projects/main_multi_timeframe.py`, allowing rapid local experimentation.
- Once satisfied with new parameters (RSI thresholds, SMA lengths, symbols), port them back to Lean knowing the behavior matches the MCP/Streamlit tooling.
- Use the saved Plotly chart (`graphs/multi_timeframe_signals.html`) and `trades_df` export to feed the broader docs/reports workflows.